In [ ]:
%%capture
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q langgraph chromadb sentence-transformers datasets trl gradio requests transformers
!pip install armoriq-sdk
print("✅ All packages installed")


In [ ]:
import armoriq_sdk
print(armoriq_sdk.__version__)

In [ ]:
import os

try:
    from armoriq_sdk import ArmorIQClient, SessionOptions
    from armoriq_sdk.exceptions import (
        IntentMismatchException,
        PolicyBlockedException,
        PolicyHoldException,
        TokenExpiredException,
        ConfigurationException,
    )
    ARMORIQ_AVAILABLE = True
except ImportError:
    ARMORIQ_AVAILABLE = False
    print("⚠️  armoriq-sdk not found — will use Python fallback governance")

# ── Credentials ────────────────────────────────────────────────────────
# Option A: Colab Secrets (recommended) → add ARMORIQ_API_KEY in the 🔑 panel
# Option B: Environment variable
try:
    from google.colab import userdata
    ARMORIQ_API_KEY = userdata.get("ARMOR_API_KEY")
except Exception:
    ARMORIQ_API_KEY = os.environ.get("ARMOR_API_KEY", "")

# ── Config ──────────────────────────────────────────────────────────────
ARMORIQ_MCP   = "armorguardian-mcp"       # Register this name on platform.armoriq.ai
ARMORIQ_USER  = "agent@armorguardian.ai"  # Default agent identity for audit logs

# ── Initialise client ───────────────────────────────────────────────────
if ARMORIQ_AVAILABLE and ARMORIQ_API_KEY:
    armoriq_client = ArmorIQClient(api_key=ARMORIQ_API_KEY)
    print("✅ ArmorIQ client initialised")
else:
    armoriq_client = None
    print("⚠️  ArmorIQ client NOT initialised — Python fallback will be used")
    print("    → Set ARMORIQ_API_KEY in Colab Secrets to enable full enforcement")


In [ ]:
import re, json, gc
from typing import TypedDict, List, Dict, Any, Optional

import torch
import chromadb
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, END

print("✅ Imports ready")
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from unsloth import FastLanguageModel

BASE_MODEL   = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LEN  = 2048
ADAPTER_PATH = "/content/guardian-gemma-adapter"
DO_FINETUNE  = False  # Set True for ~100 steps LoRA fine-tune

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    max_seq_length=MAX_SEQ_LEN,
)
print("✅ Model ready")


In [ ]:
if DO_FINETUNE:
    from datasets import load_dataset
    from trl import SFTTrainer
    from transformers import TrainingArguments

    ENTITY_TYPE_MAP = {
        "FIRSTNAME":"NAME","LASTNAME":"NAME","FULLNAME":"NAME","GIVENNAME":"NAME","FAMILYNAME":"NAME",
        "EMAIL":"EMAIL","EMAIL_ADDRESS":"EMAIL","EMAILADDRESS":"EMAIL",
        "PHONE":"PHONE","PHONENUMBER":"PHONE","TELEPHONENO":"PHONE",
        "SSN":"SSN","SOCIALSECURITYNUMBER":"SSN","SIN":"SSN",
        "CREDITCARDNUMBER":"CREDIT_CARD","CREDITCARD":"CREDIT_CARD",
        "IBAN":"IBAN","BANKACCOUNT":"BANK_ACCOUNT","ACCOUNTNUMBER":"BANK_ACCOUNT",
        "ADDRESS":"LOCATION","CITY":"LOCATION","STATE":"LOCATION","ZIPCODE":"LOCATION",
        "DATE":"DATE","DATEOFBIRTH":"DATE_OF_BIRTH","DOB":"DATE_OF_BIRTH",
        "AADHAAR":"AADHAAR","AADHAARNUMBER":"AADHAAR",
        "PAN":"PAN","PANNUMBER":"PAN",
    }

    def format_example(batch):
        out=[]
        for src, mask in zip(batch["source_text"], batch["privacy_mask"]):
            if isinstance(mask, str):
                try: mask=json.loads(mask)
                except: mask=[]
            entities=[]
            if isinstance(mask, list):
                for it in mask:
                    if not isinstance(it, dict): continue
                    raw=str(it.get("label", it.get("type","UNKNOWN"))).upper()
                    et=ENTITY_TYPE_MAP.get(raw, raw)
                    val=str(it.get("value","")).strip()
                    if val: entities.append({"type":et,"value":val})
            out_json=json.dumps({"entities":entities}, ensure_ascii=False)
            prompt = (
                "<start_of_turn>user\n"
                "Identify all PII entities in the text below. Return ONLY valid JSON.\n\n"
                f"Text: {src}\n"
                "<end_of_turn>\n"
                "<start_of_turn>model\n"
                f"{out_json}\n"
                "<end_of_turn>"
            )
            out.append(prompt)
        return {"text": out}

    ds = load_dataset("ai4privacy/pii-masking-200k", split="train[:1000]")
    ds = ds.map(format_example, batched=True, remove_columns=ds.column_names)

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=ds,
        dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
        args=TrainingArguments(
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            warmup_steps=10, max_steps=100, learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
            lr_scheduler_type="linear", seed=42,
            output_dir="./training_outputs", report_to="none",
        ),
    )
    print("🚀 Fine-tuning (100 steps)...")
    trainer.train()
    model.save_pretrained(ADAPTER_PATH)
    tokenizer.save_pretrained(ADAPTER_PATH)
    print("✅ Adapter saved to:", ADAPTER_PATH)
    del trainer; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
else:
    print("ℹ️  Skipping fine-tune. Set DO_FINETUNE=True to enable.")


In [ ]:
try:
    if os.path.exists(ADAPTER_PATH):
        print("Loading adapter:", ADAPTER_PATH)
        model.load_adapter(ADAPTER_PATH)
except Exception as e:
    print("Adapter load skipped:", e)

FastLanguageModel.for_inference(model)
print("✅ Inference mode enabled")


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

VISION_MODEL_ID = "google/gemma-4-E2B-it"
VISION_ENABLED  = False   # Set True if you have enough VRAM (~16GB)

if VISION_ENABLED:
    try:
        processor = AutoProcessor.from_pretrained(VISION_MODEL_ID)
        vision_model = AutoModelForImageTextToText.from_pretrained(
            VISION_MODEL_ID, device_map="auto", torch_dtype=torch.float16
        )
        vision_model.eval()
        print("✅ Vision model loaded")
    except Exception as e:
        VISION_ENABLED = False
        print(f"⚠️ Vision load failed: {e} — OCR will use mock text")
else:
    print("ℹ️  Vision disabled — OCR returns mock text. Set VISION_ENABLED=True to activate.")


In [ ]:
from PIL import Image

def extract_text_from_image(pil_image: Image.Image) -> str:
    """OCR using Gemma vision if available; otherwise returns a safe mock."""
    if not VISION_ENABLED:
        return (
            "Patient: John Doe\n"
            "SSN: 123-45-6789\n"
            "IBAN: DE89370400440532013000\n"
            "Diagnosis: Type 2 Diabetes\n"
            "Referring physician: Dr. Anna Schmidt\n"
            "Contact: john.doe@hospital.eu\n"
        )
    try:
        prompt = (
            "<start_of_turn>user\n<image>\n"
            "You are an OCR assistant. Extract ALL text from this document image exactly as it appears."
            " Preserve structure. Output only the extracted text.\n"
            "<end_of_turn>\n<start_of_turn>model\n"
        )
        inputs = processor(text=prompt, images=pil_image, return_tensors="pt")
        if torch.cuda.is_available():
            inputs = {k:v.to("cuda") for k,v in inputs.items()}
        with torch.no_grad():
            out = vision_model.generate(**inputs, max_new_tokens=1024, do_sample=False)
        decoded = processor.decode(out[0], skip_special_tokens=True)
        return decoded.split("model")[-1].strip() if "model" in decoded else decoded.strip()
    except Exception as e:
        print("⚠️ Vision OCR failed; using fallback. Error:", e)
        return "Patient: John Doe\nSSN: 123-45-6789\nIBAN: DE89370400440532013000\n"


In [ ]:
COMPLIANCE_DOCS = [
    {"id":"gdpr_art4",  "framework":"GDPR",  "text":"GDPR Article 4: Personal data means any information relating to an identified or identifiable natural person, including name, identification number, location data, online identifier."},
    {"id":"gdpr_art9",  "framework":"GDPR",  "text":"GDPR Article 9: Special categories include racial/ethnic origin, political opinions, religious beliefs, genetic/biometric data, health data, sex life data."},
    {"id":"gdpr_art17", "framework":"GDPR",  "text":"GDPR Article 17: Right to erasure (right to be forgotten) without undue delay under certain conditions."},
    {"id":"gdpr_art44", "framework":"GDPR",  "text":"GDPR Article 44: Any transfer of personal data to a third country shall only take place if the transfer complies with adequacy conditions."},
    {"id":"hipaa_sh",   "framework":"HIPAA", "text":"HIPAA Safe Harbor (45 CFR 164.514(b)) requires removal of 18 identifiers including names, geographic data, dates, SSN, medical records, account numbers, biometric identifiers."},
    {"id":"hipaa_mn",   "framework":"HIPAA", "text":"HIPAA Minimum Necessary (45 CFR 164.502(b)): limit use and disclosure to the minimum necessary to accomplish the intended purpose."},
    {"id":"pci_req3",   "framework":"PCI",   "text":"PCI-DSS Req 3: Protect stored cardholder data; PAN must be rendered unreadable; mask all but last four digits when displayed."},
    {"id":"pci_req4",   "framework":"PCI",   "text":"PCI-DSS Req 4: Do not store sensitive authentication data after authorization (track data, CVV/CVC, PIN blocks)."},
    {"id":"dpdp_s2",    "framework":"DPDP",  "text":"India DPDP Act 2023: Personal data includes Aadhaar, PAN, financial account details, health data, and biometric data of identifiable individuals."},
    {"id":"dpdp_s4",    "framework":"DPDP",  "text":"India DPDP Act 2023 Section 4: Processing generally requires consent of the Data Principal for a lawful purpose."},
    {"id":"dpdp_s6",    "framework":"DPDP",  "text":"India DPDP Act 2023 Section 6: Consent must be free, specific, informed, unconditional, and unambiguous with a clear affirmative action."},
    {"id":"dpdp_s9",    "framework":"DPDP",  "text":"India DPDP Act 2023 Section 9: Processing of children's data (<18) requires verifiable parental consent. Penalty up to ₹200 Cr."},
    {"id":"dpdp_s16",   "framework":"DPDP",  "text":"India DPDP Act 2023 Section 16: Cross-border transfer of personal data subject to Central Government notification and safeguards."},
    {"id":"uk_gdpr_a22","framework":"UK_GDPR","text":"UK GDPR Article 22: Data subjects have the right not to be subject to solely automated decisions with significant effects."},
    {"id":"appi_20",    "framework":"APPI",  "text":"Japan APPI Article 20(2): Handling of Special Care-Required Personal Information requires opt-in prior consent from the data subject."},
]

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("guardian_compliance")
except Exception:
    pass
collection = chroma_client.create_collection("guardian_compliance", metadata={"hnsw:space":"cosine"})
embedder   = SentenceTransformer("all-MiniLM-L6-v2")

for doc in COMPLIANCE_DOCS:
    emb = embedder.encode(doc["text"]).tolist()
    collection.add(ids=[doc["id"]], documents=[doc["text"]],
                   embeddings=[emb], metadatas=[{"framework": doc["framework"]}])

print("✅ RAG ready:", len(COMPLIANCE_DOCS), "docs")


In [ ]:
# Regulation → articles that map to each framework (for audit log)
FRAMEWORK_ARTICLES = {
    "GDPR":    ["Art.4 (definition)", "Art.9 (special categories)", "Art.17 (erasure)", "Art.25 (privacy by design)", "Art.44 (transfers)"],
    "DPDP":    ["Sec.4 (lawful purpose)", "Sec.6 (consent)", "Sec.9 (children)", "Sec.12 (rights)", "Sec.16 (cross-border)"],
    "HIPAA":   ["45 CFR 164.514 (safe harbor)", "45 CFR 164.502 (minimum necessary)"],
    "PCI":     ["Req.3 (stored data)", "Req.4 (auth data)"],
    "UK_GDPR": ["Art.4", "Art.9", "Art.22 (automated decisions)", "Art.44"],
    "APPI":    ["Art.17 (purpose)", "Art.20(2) (sensitive data)", "Art.27 (third party)", "Art.28 (cross-border)"],
}

# Entity types to redact per framework
FRAMEWORK_REDACT_TYPES = {
    "GDPR": [
        "NAME", "EMAIL", "PHONE", "DATE_OF_BIRTH",
        "ADDRESS", "LOCATION", "NATIONAL_ID",
        "IBAN", "BANK_ACCOUNT", "CREDIT_CARD", "MASKED_CARD",
        "TAX_ID", "SSN",
    ],
    # Update this in the FRAMEWORK_REDACT_TYPES dict:
    "DPDP": [
        "AADHAAR", "PAN", "NAME", "EMAIL", "PHONE",
        "CREDIT_CARD", "BANK_ACCOUNT", "DATE_OF_BIRTH",
        "PASSPORT", "DRIVING_LICENSE", "VOTER_ID",     # ← add these
        "CARD_EXPIRY", "CVV",                           # ← add these
    ],
    # Update FRAMEWORK_REDACT_TYPES["HIPAA"]:
    "HIPAA": [
        "NAME", "SSN", "DATE_OF_BIRTH", "PHONE", "EMAIL",
        "ADDRESS", "LOCATION", "PATIENT_ID",
        "INSURANCE_ID", "MRN", "NPI", "GROUP_NUMBER",   # ← add these
        "BANK_ACCOUNT", "CREDIT_CARD","EMPLOYEE",
    ],
    "PCI":     ["CREDIT_CARD", "BANK_ACCOUNT", "IBAN"],
    "UK_GDPR": ["NAME", "EMAIL", "PHONE", "SSN", "LOCATION"],
    "APPI":    ["NAME", "EMAIL", "PHONE", "LOCATION", "DATE_OF_BIRTH"],
}

# ArmorIQ intent plan templates per framework
FRAMEWORK_PLANS = {
    "GDPR": {
        "goal": "Extract, classify and redact PII under EU GDPR compliance",
        "regulation": "EU General Data Protection Regulation",
        "steps": [
            {"action": "extract_pii",        "mcp": ARMORIQ_MCP, "params": {"framework": "GDPR"}},
            {"action": "enforce_perimeter",  "mcp": ARMORIQ_MCP, "params": {"block_cross_border": True}},
            {"action": "redact_pii",         "mcp": ARMORIQ_MCP, "params": {"types": FRAMEWORK_REDACT_TYPES["GDPR"]}},
            {"action": "audit_log",          "mcp": ARMORIQ_MCP, "params": {"regulation": "GDPR", "articles": FRAMEWORK_ARTICLES["GDPR"]}},
        ]
    },
    "DPDP": {
        "goal": "Extract, classify and redact PII under India DPDP Act 2023",
        "regulation": "India Digital Personal Data Protection Act 2023",
        "steps": [
            {"action": "extract_pii",        "mcp": ARMORIQ_MCP, "params": {"framework": "DPDP"}},
            {"action": "verify_consent",     "mcp": ARMORIQ_MCP, "params": {"required": True}},
            {"action": "enforce_perimeter",  "mcp": ARMORIQ_MCP, "params": {"block_cross_border": True, "block_child_data": True}},
            {"action": "redact_pii",         "mcp": ARMORIQ_MCP, "params": {"types": FRAMEWORK_REDACT_TYPES["DPDP"]}},
            {"action": "audit_log",          "mcp": ARMORIQ_MCP, "params": {"regulation": "DPDP", "articles": FRAMEWORK_ARTICLES["DPDP"]}},
        ]
    },
    "HIPAA": {
        "goal": "Extract, classify and redact PHI under HIPAA Safe Harbor",
        "regulation": "HIPAA (45 CFR 164.514)",
        "steps": [
            {"action": "extract_pii",        "mcp": ARMORIQ_MCP, "params": {"framework": "HIPAA"}},
            {"action": "enforce_perimeter",  "mcp": ARMORIQ_MCP},
            {"action": "redact_pii",         "mcp": ARMORIQ_MCP, "params": {"types": FRAMEWORK_REDACT_TYPES["HIPAA"]}},
            {"action": "audit_log",          "mcp": ARMORIQ_MCP, "params": {"regulation": "HIPAA", "articles": FRAMEWORK_ARTICLES["HIPAA"]}},
        ]
    },
    "PCI": {
        "goal": "Extract, classify and redact cardholder data under PCI-DSS",
        "regulation": "PCI-DSS v4.0",
        "steps": [
            {"action": "extract_pii",        "mcp": ARMORIQ_MCP, "params": {"framework": "PCI"}},
            {"action": "enforce_perimeter",  "mcp": ARMORIQ_MCP},
            {"action": "redact_pii",         "mcp": ARMORIQ_MCP, "params": {"types": FRAMEWORK_REDACT_TYPES["PCI"]}},
            {"action": "audit_log",          "mcp": ARMORIQ_MCP, "params": {"regulation": "PCI-DSS", "articles": FRAMEWORK_ARTICLES["PCI"]}},
        ]
    },
}
print("✅ ArmorIQ framework plans defined")


In [ ]:
def detect_region_and_domain(text: str) -> Dict[str, str]:
    """Detect jurisdiction and domain from document text heuristics."""
    t = text.lower()
    region = "GLOBAL"
    domain = "GENERAL"

    if any(k in t for k in ["aadhaar", "pan ", "upi", "india", "bengaluru", "bangalore", "rupee", "₹"]):
        region = "INDIA"
    elif any(k in t for k in ["gdpr", "europe", "eu citizen", "iban", "berlin", ".eu", "dsgvo"]):
        region = "EU"
    elif any(k in t for k in ["uk ", "united kingdom", "ico ", "british", ".co.uk"]):
        region = "UK"
    elif any(k in t for k in ["japan", "japanese", "appi", "マイナンバー", "個人情報"]):
        region = "JAPAN"

    if any(k in t for k in ["patient", "diagnosis", "hospital", "dr.", "doctor", "medical", "phi", "prescription", "clinic"]):
        domain = "HEALTHCARE"
    elif any(k in t for k in ["credit card", "pan:", "cvv", "iban", "account", "payment", "invoice", "transaction"]):
        domain = "FINANCE"

    # Map region → primary compliance framework
    framework_map = {
        "INDIA":  "DPDP",
        "EU":     "GDPR",
        "UK":     "GDPR",   # UK GDPR same enforcement path for now
        "JAPAN":  "GDPR",   # APPI fallback to GDPR path
        "GLOBAL": "HIPAA" if domain == "HEALTHCARE" else "PCI" if domain == "FINANCE" else "GDPR",
    }

    return {"region": region, "domain": domain, "framework": framework_map.get(region, "GDPR")}


In [ ]:
# ── spaCy NER setup (run once) ───────────────────────────────────────────
import subprocess, sys

try:
    import spacy
    try:
        nlp = spacy.load("en_core_web_sm")
        print("✅ spaCy en_core_web_sm ready")
    except OSError:
        print("⬇️  Downloading en_core_web_sm...")
        subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
                       check=True, capture_output=True)
        nlp = spacy.load("en_core_web_sm")
        print("✅ spaCy en_core_web_sm ready")
    SPACY_AVAILABLE = True
except Exception as e:
    SPACY_AVAILABLE = False
    nlp = None
    print(f"⚠️  spaCy not available ({e}) — name extraction will rely on Gemma only")


# ── Tokens that must NOT appear in any extracted name ────────────────────
_SPACY_BLOCKLIST = {
    # Marital / relationship labels
    "married", "unmarried", "single", "divorced", "widow", "widower",
    "nominee", "holder", "wife", "husband", "spouse", "son", "daughter",
    "father", "mother", "brother", "sister", "guardian", "relationship",
    # Form field labels
    "name", "card", "number", "expiry", "date", "address", "branch",
    "account", "email", "mobile", "alternate", "contact", "signature",
    "type", "status", "gender", "male", "female", "other",
    # Indian geography (spaCy often tags states/cities as PERSON)
    "haryana", "punjab", "delhi", "mumbai", "kerala", "gurgaon",
    "bangalore", "chennai", "kolkata", "hyderabad", "india", "sector",
    "street", "road", "nagar", "vihar", "colony",
    # Financial / insurance labels
    "pvt", "ltd", "co", "corp", "inc", "llp", "bank", "insurance",
    "policy", "premium", "claim", "upi", "ifsc",
    # Medical labels
    "blood", "group", "allergy", "diagnosis", "physician", "doctor",
    "patient", "ward", "bed", "dept", "icu","allergies", "known", "existing",
    "conditions", "hypertension",
    "diabetes", "blood", "admission", "consent", "discharge",
    # ── 1. Add to _SPACY_BLOCKLIST ───────────────────────────────────────────
    "closing", "opening", "balance", "statement", "period",
    "debit", "credit", "salary", "transfer", "sepa",
    "transaction", "currency", "amount", "total",
    "gmbh", "sarl", "ag", "kg", "plc", "bv", "nv", "sa", "srl",
    "bank", "nordbank", "telekom", "amazon",
}


def extract_names_spacy(text: str) -> List[Dict[str, str]]:
    """
    Extract PERSON entities using spaCy NER with multi-layer noise filtering.

    Filters applied (in order):
    1. Must have ≥ 2 tokens (first + last name minimum)
    2. No token appears in the blocklist
    3. Max 4 tokens (prevents OCR run-ons like 'Rahul Arora Nominee Wife')
    4. Every token must start with an uppercase letter (proper noun check)
    5. No all-caps tokens longer than 2 chars (abbreviations / labels like PAN, UPI)
    6. Value must be ≥ 5 chars total (rejects 'Mr', 'Dr' stubs)
    7. No digit characters (rejects OCR artefacts like 'Arora 2026')
    """
    if not SPACY_AVAILABLE or nlp is None:
        return []

    doc     = nlp(text)
    seen    = set()
    entities = []

    for ent in doc.ents:
        if ent.label_ != "PERSON":
            continue

        val    = ent.text.strip()
        tokens = val.split()

        # 1. Minimum two tokens
        if len(tokens) < 2:
            continue

        # 2. No blocklisted token
        if any(t.lower() in _SPACY_BLOCKLIST for t in tokens):
            continue

        # 3. Max 4 tokens
        if len(tokens) > 4:
            continue

        # 4. Every token starts with uppercase
        if not all(t[0].isupper() for t in tokens if t):
            continue

        # 5. No all-caps token longer than 2 chars
        if any(t.isupper() and len(t) > 2 for t in tokens):
            continue

        # 6. Total length ≥ 5 chars
        if len(val) < 5:
            continue

        # 7. No digits anywhere in the name
        if any(c.isdigit() for c in val):
            continue
        # Filter 8 — every token must be at least 3 chars (blocks 'Rrse', 'Mr', 'T')
        if any(len(t) < 3 for t in tokens):
            continue
        if val not in seen:
            seen.add(val)
            entities.append({"type": "NAME", "value": val})
        # 10. Vocabulary plausibility — reject OCR gibberish tokens
        #     Real names exist in spaCy vocab; 'Fecord', 'Summaly', 'Rrse' do not
        def _is_vocab_plausible(tok: str) -> bool:
            lex = nlp.vocab[tok.lower()]
            # prob = -inf for unknown words; real English words/names > -20
            return lex.prob > -20

        # At least the last token (surname) must be vocabulary-plausible
        # First names can be rare/foreign so only enforce on surname
        if not _is_vocab_plausible(tokens[-1]):
            continue

        if val not in seen:
            seen.add(val)
            entities.append({"type": "NAME", "value": val})
    return entities


# ── Quick self-test ───────────────────────────────────────────────────────
if SPACY_AVAILABLE:
    _test = (
        "Patient Rahul Arora (DOB: 14-08-1990) was admitted by Dr. Priya Menon. "
        "Emergency contact: Meena Nair (Wife). Policy Holder Name HDFC Bank Haryana."
    )
    _result = extract_names_spacy(_test)
    print(f"   Self-test → {[e['value'] for e in _result]}")
    # Expected: ['Rahul Arora', 'Priya Menon', 'Meena Nair']
    # Should NOT contain: 'Holder Name', 'HDFC Bank', 'Haryana', 'Wife'

In [ ]:

def gemma_pii_extract(text: str) -> List[Dict[str, str]]:
    """Use Gemma 4 2B to extract PII; falls back to regex on failure.
    Chunks long documents so Gemma never sees more than 800 chars at once.
    Adds spaCy NER as a final pass for NAME entities.
    """
    import re as _re

    MAX_CHUNK = 800

    chunks = []
    if len(text) <= MAX_CHUNK:
        chunks = [text]
    else:
        step = MAX_CHUNK - 100
        for i in range(0, len(text), step):
            chunks.append(text[i : i + MAX_CHUNK])

    all_entities: List[Dict[str, str]] = []
    seen: set = set()

    def _dedup_add(new_entities):
        for e in new_entities:
            key = (e.get("type", ""), e.get("value", ""))
            if key not in seen and e.get("value", "").strip():
                seen.add(key)
                all_entities.append(e)

    def _gemma_chunk(chunk: str) -> List[Dict[str, str]]:
        messages = [{
            "role": "user",
            "content": [{
                "type": "text",
                "text": (
                    "Extract all PII entities from this text.\n"
                    "Return ONLY valid JSON with an \"entities\" array, "
                    "each item having \"type\" and \"value\". "
                    "Types: NAME EMAIL PHONE SSN AADHAAR PAN CREDIT_CARD "
                    "IBAN BANK_ACCOUNT DATE_OF_BIRTH LOCATION PATIENT_ID.\n\n"
                    f"TEXT:\n{chunk}"
                )
            }]
        }]
        inputs = text_tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt"
        )
        device = "cuda" if torch.cuda.is_available() else "cpu"
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=256, do_sample=False,
                repetition_penalty=1.1,
                eos_token_id=text_tokenizer.eos_token_id,
                pad_token_id=text_tokenizer.eos_token_id,
            )
        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        response = text_tokenizer.decode(generated_tokens, skip_special_tokens=True)
        clean = response.replace("```json", "").replace("```", "").strip()
        match = _re.search(r'\{.*\}', clean, _re.DOTALL)
        if not match:
            raise ValueError(f"No JSON object in Gemma response: {clean[:80]}")
        return json.loads(match.group(0)).get("entities", [])

    # ── 1. Gemma chunks ───────────────────────────────────────────────────
    gemma_ok = 0
    for i, chunk in enumerate(chunks):
        try:
            _dedup_add(_gemma_chunk(chunk))
            gemma_ok += 1
        except Exception as e:
            print(f"⚠️  Gemma chunk {i+1}/{len(chunks)} failed: {e} — regex covers this chunk")
            _dedup_add(fallback_regex_detector(chunk))

    # ── 2. Regex safety net over full text ───────────────────────────────
    _dedup_add(fallback_regex_detector(text))

    # ── 3. spaCy NER for names (runs on full text, CPU-fast) ─────────────
    name_entities = extract_names_spacy(text)
    _dedup_add(name_entities)
    if name_entities:
        print(f"   🔤 spaCy found {len(name_entities)} names: "
              f"{[e['value'] for e in name_entities]}")

    status = "✅ Gemma" if gemma_ok == len(chunks) else f"⚠️  Gemma {gemma_ok}/{len(chunks)} chunks"
    print(f"{status} + regex + spaCy → {len(all_entities)} PII entities  "
          f"({len(chunks)} chunk{'s' if len(chunks)>1 else ''})")
    return all_entities


def fallback_regex_detector(text: str) -> List[Dict[str, str]]:
    """Regex-based PII detection — covers Indian phone formats,
    emails, credit cards, Aadhaar, PAN, dates, bank accounts."""
    patterns = {
        "EMAIL":         r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "PHONE":         r"(?:\+91[\s\-]?)?[6-9]\d{4}[\s\-]?\d{5}\b",
        "SSN":           r"\b\d{3}-\d{2}-\d{4}\b",
        "IBAN":          r"\b[A-Z]{2}\d{2}[A-Z0-9]{10,30}\b",
        "CREDIT_CARD":   r"\b\d{4}[\s\-]?\d{4}[\s\-]?\d{4}[\s\-]?\d{4}\b",
        "AADHAAR":       r"\b\d{4}[\s\-]?\d{4}[\s\-]?\d{4}\b",
        "PAN":           r"\b[A-Z]{5}\d{4}[A-Z]\b",
        "DATE_OF_BIRTH": r"(?<!\w)(?:\d{1,2}[-/]\d{1,2}[-/]\d{2,4}|\d{4}[-/]\d{1,2}[-/]\d{1,2})(?!\w)",
        "BANK_ACCOUNT":  r"\b\d{9,18}\b",
        "PASSPORT":       r"\b[A-Z]\d{7}\b",
        "DRIVING_LICENSE": r"\b[A-Z]{2}-\d{2}-\d{4}\d+\b",
        "VOTER_ID":       r"\b[A-Z]{2,3}/\d{2}/\d{4}/\d+\b",
        "INSURANCE_ID":  r"\b[A-Z]{2,4}[-\s]?[A-Z]{0,2}[-\s]?\d{6,12}\b",
        "MRN":           r"\b(?:MRN|GV|MR)[-:\s]?[\w\-]{6,20}\b",
        "NPI": r"\bNPI:?\s*[\d/\\|]{9,12}\b",
        "GROUP_NUMBER":  r"\bGRP[-\s]?\d{4}[-\s]?[A-Z]{2}\b",
        "EMPLOYER": r"\b[A-Z][a-z]+(?: [A-Z][a-z]+)* (?:School District|Independent School|Hospital|University|Medical Center|Insurance|Bank|Corp|Inc|LLC|Institute|College)\b",        "INSURANCE_ID":  r"\b[A-Z]{2,5}[-\s]?[A-Z]{0,3}[-\s]?\d{6,12}\b",
        "TAX_ID":    r"\b[A-Z]{2}-TAX-\d{8,12}\b",
        "NATIONAL_ID": r"\b\d{10}\s*\([A-Za-z]+\)",        # 1234567890 (Personalausweis)
        "IBAN":      r"\b[A-Z]{2}\d{2}[\s]?\d{4}[\s]?\d{4}[\s]?\d{4}[\s]?\d{4}[\s]?\d{2}\b",
        "MASKED_CARD": r"\b\d{4}\s+\*+\s+\*+\s+\d{4}\b",  # 4532 **** ** 7891
    }
    entities, seen_vals = [], set()
    for etype, pattern in patterns.items():
        for m in re.finditer(pattern, text):
            val = m.group(0).strip()
            if val and val not in seen_vals:
                seen_vals.add(val)
                entities.append({"type": etype, "value": val})
    return entities

In [ ]:
def rag_retrieve(text: str, k: int = 3) -> List[Dict]:
    """Retrieve top-k compliance articles from ChromaDB relevant to the document."""
    q_emb = embedder.encode(text).tolist()
    res   = collection.query(
        query_embeddings=[q_emb], n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    ids   = res.get("ids", [[]])[0]
    docs  = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    dists = res.get("distances", [[]])[0]
    return [
        {"id": ids[i], "text": docs[i], "framework": metas[i].get("framework","?"),
         "distance": dists[i]}
        for i in range(len(docs))
    ]


In [ ]:
def python_fallback_governance(framework: str, entities: List[Dict]) -> List[str]:
    """Pure-Python mirror of ArmorIQ policy rules — used when SDK is unavailable."""
    redact_types = FRAMEWORK_REDACT_TYPES.get(framework, ["NAME","EMAIL","PHONE"])
    vals = [e["value"] for e in entities if e.get("type") in redact_types]
    print(f"⚠️  Python fallback governance — {len(vals)} values selected for redaction")
    return vals


In [ ]:
import hashlib

def armoriq_decide(
    framework: str,
    region: str,
    domain: str,
    user_role: str,
    entities: List[Dict],
    raw_text: str,
) -> tuple:
    """
    ArmorIQ-powered policy enforcement.
    Returns: (values_to_redact: list, token_id: str, enforcement_log: list)
    """
    enforcement_log = []

    # ── No client available → Python fallback ──────────────────────────
    if not armoriq_client:
        vals = python_fallback_governance(framework, entities)
        enforcement_log.append({"step": "fallback", "reason": "no_armoriq_client"})
        return vals, "NO_TOKEN", enforcement_log

    plan_cfg = FRAMEWORK_PLANS.get(framework, FRAMEWORK_PLANS["GDPR"])

    try:
        # ── Step 1: Capture intent plan ─────────────────────────────────
        print(f"🔐 ArmorIQ: capturing plan for {framework}...")
        plan = armoriq_client.capture_plan(
            llm="gemma-4-2b",
            prompt=(
                f"Process PII document under {framework} compliance. "
                f"User role: {user_role}. Region: {region}. Domain: {domain}."
            ),
            plan={
                "goal": plan_cfg["goal"],
                "steps": plan_cfg["steps"],
            }
        )

        # ── Step 2: Mint intent token (5 min validity) ──────────────────
        token     = armoriq_client.get_intent_token(plan, validity_seconds=300)
        # ✅ Fix — strip the repr prefix if present
        raw_id   = getattr(token, "id", None) or str(token)
        token_id = raw_id.replace("token_id='", "").replace("'", "").strip()[:24]
        enforcement_log.append({
            "step": "intent_locked",
            "token_id": token_id,
            "framework": framework,
            "regulation": plan_cfg["regulation"],
        })
        print(f"✅ Intent token minted: {token_id}")

        # ── Step 3: Gate — extract_pii ───────────────────────────────────
        armoriq_client.invoke(
            mcp=ARMORIQ_MCP, action="extract_pii", intent_token=token,
            params={"entities_count": len(entities), "framework": framework},
            user_email=ARMORIQ_USER,
        )

        enforcement_log.append({"step": "extract_pii", "status": "allowed", "count": len(entities)})

        # ── Step 4: Gate — enforce_perimeter (prompt injection block) ───
        text_hash = hashlib.sha256(raw_text.encode()).hexdigest()[:16]
        armoriq_client.invoke(
            mcp=ARMORIQ_MCP, action="enforce_perimeter", intent_token=token,
            params={"text_hash": text_hash, "framework": framework},
            user_email=ARMORIQ_USER,
        )
        enforcement_log.append({"step": "enforce_perimeter", "status": "passed", "text_hash": text_hash})

        # ── Step 5: Derive redaction list from framework policy ──────────
        redact_types    = FRAMEWORK_REDACT_TYPES.get(framework, ["NAME","EMAIL","PHONE"])
        values_to_redact = [e["value"] for e in entities if e.get("type") in redact_types]

        # ── Step 6: Gate — redact_pii ────────────────────────────────────
        armoriq_client.invoke(
            mcp=ARMORIQ_MCP, action="redact_pii", intent_token=token,
            params={"redacted_count": len(values_to_redact), "types": redact_types},
            user_email=ARMORIQ_USER,
        )
        enforcement_log.append({"step": "redact_pii", "status": "allowed", "redacted_count": len(values_to_redact)})

        # ── Step 7: Gate — audit_log ─────────────────────────────────────
        armoriq_client.invoke(
            mcp=ARMORIQ_MCP, action="audit_log", intent_token=token,
            params={
                "regulation":        plan_cfg["regulation"],
                "articles":          FRAMEWORK_ARTICLES.get(framework, []),
                "entities_detected": len(entities),
                "entities_redacted": len(values_to_redact),
                "user_role":         user_role,
                "region":            region,
                "domain":            domain,
            },
            user_email=ARMORIQ_USER,
        )
        enforcement_log.append({
            "step": "audit_log", "status": "logged",
            "regulation": framework,
            "articles": FRAMEWORK_ARTICLES.get(framework, []),
        })

        print(f"✅ ArmorIQ enforcement complete | Token: {token_id} | Redacting: {len(values_to_redact)} values")
        return values_to_redact, token_id, enforcement_log

    # ── Prompt injection / intent drift detected ─────────────────────────
    except IntentMismatchException as e:
        msg = str(e)
        enforcement_log.append({"step": "BLOCKED", "reason": "intent_mismatch", "detail": msg})
        print(f"🚨 INTENT MISMATCH — Prompt injection attempt BLOCKED: {msg}")
        return [], "BLOCKED_INJECTION", enforcement_log

    # ── Policy explicitly denies this action ─────────────────────────────
    except PolicyBlockedException as e:
        msg = str(e)
        enforcement_log.append({"step": "BLOCKED", "reason": "policy_blocked", "detail": msg})
        print(f"🔒 POLICY BLOCKED — Action not permitted: {msg}")
        return [], "POLICY_BLOCKED", enforcement_log

    # ── Requires human approval before proceeding ────────────────────────
    except PolicyHoldException as e:
        msg = str(e)
        enforcement_log.append({"step": "HOLD", "reason": "human_approval_required", "detail": msg})
        print(f"⏸️  POLICY HOLD — Human approval required: {msg}")
        # Proceed with fallback (partial redaction) until approval arrives
        vals = python_fallback_governance(framework, entities)
        return vals, "ON_HOLD", enforcement_log

    # ── Token expired mid-session — re-issue once ────────────────────────
    except TokenExpiredException:
        enforcement_log.append({"step": "RETRY", "reason": "token_expired"})
        print("⚠️  Intent token expired — re-issuing...")
        return armoriq_decide(framework, region, domain, user_role, entities, raw_text)

    # ── General ArmorIQ error — use Python fallback ──────────────────────
    except Exception as e:
        enforcement_log.append({"step": "FALLBACK", "reason": str(e)})
        print(f"⚠️  ArmorIQ error ({type(e).__name__}): {e} — using Python fallback")
        vals = python_fallback_governance(framework, entities)
        return vals, "FALLBACK", enforcement_log


In [ ]:
def redact_text(raw: str, values_to_redact: List[str]) -> tuple:
    sanitized, log = raw, []
    for v in sorted(set(values_to_redact), key=len, reverse=True):
        if not v or v not in sanitized:
            continue
        sanitized = sanitized.replace(v, "[REDACTED]")
        log.append({"redacted_value": v, "replacement": "[REDACTED]"})
    return sanitized, log


def build_trust_report(
    entities: List[Dict],
    redacted_values: List[str],
    retrieved: List[Dict],
    framework: str,
    token_id: str = "N/A",
    enforcement_log: List[Dict] = None,
) -> Dict:
    detected          = len(entities)
    entities_redacted = len(redacted_values)                          # ← was missing
    redact_types      = FRAMEWORK_REDACT_TYPES.get(framework, [])
    relevant_count    = sum(1 for e in entities if e.get("type") in redact_types)
    coverage          = entities_redacted / relevant_count if relevant_count else 1.0
    rag_bonus         = 0.1 if retrieved else 0.0
    score             = max(0.0, min(1.0, 0.6 * coverage + 0.4 + rag_bonus)) if detected else 0.9


    # Determine enforcement mode from log
    enforcement_mode = "python_fallback"
    if enforcement_log:
        steps = [e.get("step","") for e in enforcement_log]
        if "intent_locked" in steps:
            if "BLOCKED" in steps:
                enforcement_mode = "armoriq_blocked"
            elif "HOLD" in steps:
                enforcement_mode = "armoriq_hold"
            else:
                enforcement_mode = "armoriq_enforced"

    return {
        "framework":           framework,
        "regulation":          FRAMEWORK_PLANS.get(framework, {}).get("regulation", framework),
        "entities_detected":   detected,
        "entities_redacted":   entities_redacted,
        "coverage":            round(coverage, 2),
        "policy_compliance":   True,
        "enforcement_mode":    "armoriq_enforced",
        "armoriq_token_id":    token_id,
        "trust_score":         round(score, 2),
        "rag_articles": [{"id": r["id"], "framework": r.get("framework", "?")} for r in retrieved],
        "legislation_articles": FRAMEWORK_ARTICLES.get(framework, []),
    }

In [ ]:
class GuardianState(TypedDict, total=False):
    # Input
    raw_text:               str
    user_role:              str
    # Routing
    routing_context:        Dict[str, str]
    compliance_framework:   str
    # RAG
    retrieved_articles:     List[Dict[str, str]]
    # PII
    detected_entities:      List[Dict[str, str]]
    # ArmorIQ enforcement (replaces OPA fields)
    values_to_redact:       List[str]
    armoriq_token_id:       str
    armoriq_enforcement_log: List[Dict[str, Any]]
    # Redaction
    sanitized_text:         str
    redaction_log:          List[Dict[str, str]]
    # Output
    trust_report:           Dict[str, Any]


# ── Node: router ────────────────────────────────────────────────────────
def router_node(state: GuardianState):
    ctx = detect_region_and_domain(state["raw_text"])
    return {"routing_context": ctx, "compliance_framework": ctx["framework"]}


# ── Node: rag ────────────────────────────────────────────────────────────
def rag_node(state: GuardianState):
    retrieved = rag_retrieve(state["raw_text"], k=3)
    return {"retrieved_articles": retrieved}


# ── Node: pii ────────────────────────────────────────────────────────────
def pii_node(state: GuardianState):
    entities = gemma_pii_extract(state["raw_text"])
    # Augment with regex to catch any patterns Gemma missed
    seen   = {(e["type"], e["value"]) for e in entities}
    extras = [e for e in fallback_regex_detector(state["raw_text"]) if (e["type"], e["value"]) not in seen]
    return {"detected_entities": entities + extras}


# ── Node: armoriq (replaces opa_node) ───────────────────────────────────
def armoriq_node(state: GuardianState):
    fw  = state["compliance_framework"]
    ctx = state["routing_context"]

    print(f"\n🛡️  ArmorIQ node | Framework: {fw} | Region: {ctx.get('region')} | Domain: {ctx.get('domain')}")
    print(f"   Entities passed to enforcement: {len(state.get('detected_entities',[]))}")

    values, token_id, enforcement_log = armoriq_decide(
        framework    = fw,
        region       = ctx.get("region", "GLOBAL"),
        domain       = ctx.get("domain", "GENERAL"),
        user_role    = state.get("user_role", "guest"),
        entities     = state.get("detected_entities", []),
        raw_text     = state["raw_text"],
    )

    return {
        "values_to_redact":        values,
        "armoriq_token_id":        token_id,
        "armoriq_enforcement_log": enforcement_log,
    }


# ── Node: redact ─────────────────────────────────────────────────────────
def redact_node(state: GuardianState):
    sanitized, log = redact_text(state["raw_text"], state.get("values_to_redact", []))
    return {"sanitized_text": sanitized, "redaction_log": log}


# ── Node: trust ──────────────────────────────────────────────────────────
def trust_node(state: GuardianState):
    report = build_trust_report(
        entities         = state.get("detected_entities", []),
        redacted_values  = state.get("values_to_redact", []),
        retrieved        = state.get("retrieved_articles", []),
        framework        = state.get("compliance_framework", "GDPR"),
        token_id         = state.get("armoriq_token_id", "N/A"),
        enforcement_log  = state.get("armoriq_enforcement_log", []),
    )
    return {"trust_report": report}


# ── Compile graph ────────────────────────────────────────────────────────
graph = StateGraph(GuardianState)
graph.add_node("router",   router_node)
graph.add_node("rag",      rag_node)
graph.add_node("pii",      pii_node)
graph.add_node("armoriq",  armoriq_node)   # was: "opa"
graph.add_node("redact",   redact_node)
graph.add_node("trust",    trust_node)

graph.set_entry_point("router")
graph.add_edge("router",  "rag")
graph.add_edge("rag",     "pii")
graph.add_edge("pii",     "armoriq")
graph.add_edge("armoriq", "redact")
graph.add_edge("redact",  "trust")
graph.add_edge("trust",   END)

guardian_app = graph.compile()
print("✅ LangGraph compiled: router → rag → pii → armoriq → redact → trust")


In [ ]:
# armorguardian_mcp.py
# Run: pip install fastapi uvicorn
# Then: python armorguardian_mcp.py

from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
import json, re, hashlib, logging
from datetime import datetime, timezone

logging.basicConfig(level=logging.INFO)
app = FastAPI(title="ArmorGuardian MCP")

# ── Tool definitions (must match the action names used in the notebook) ──
TOOLS = [
    {
        "name": "extract_pii",
        "description": "Detect and classify PII entities in text under a given compliance framework.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "framework":       {"type": "string", "description": "GDPR | DPDP | HIPAA | PCI"},
                "entities_count":  {"type": "integer", "description": "Number of entities detected upstream"},
            },
            "required": ["framework"]
        }
    },
    {
        "name": "enforce_perimeter",
        "description": "Block prompt injection and cross-border/child-data violations.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "text_hash":          {"type": "string"},
                "framework":          {"type": "string"},
                "block_cross_border": {"type": "boolean"},
                "block_child_data":   {"type": "boolean"},
            },
            "required": ["framework"]
        }
    },
    {
        "name": "redact_pii",
        "description": "Approve the redaction list for a given framework.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "types":          {"type": "array",   "items": {"type": "string"}},
                "redacted_count": {"type": "integer"},
            },
            "required": ["types"]
        }
    },
    {
        "name": "audit_log",
        "description": "Write a tamper-evident compliance audit entry.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "regulation":        {"type": "string"},
                "articles":          {"type": "array", "items": {"type": "string"}},
                "entities_detected": {"type": "integer"},
                "entities_redacted": {"type": "integer"},
                "user_role":         {"type": "string"},
                "region":            {"type": "string"},
                "domain":            {"type": "string"},
            },
            "required": ["regulation"]
        }
    },
    {
        "name": "verify_consent",
        "description": "Check consent requirements (DPDP / GDPR).",
        "inputSchema": {
            "type": "object",
            "properties": {
                "required": {"type": "boolean"}
            }
        }
    },
]

# ── SSE helper ────────────────────────────────────────────────────────────
def sse(data: dict) -> str:
    return f"event: message\ndata: {json.dumps(data)}\n\n"

# ── Tool logic ────────────────────────────────────────────────────────────
def run_tool(name: str, args: dict) -> dict:
    ts = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

    if name == "extract_pii":
        return {
            "status": "allowed",
            "framework": args.get("framework"),
            "entities_count": args.get("entities_count", 0),
            "timestamp": ts,
        }

    elif name == "enforce_perimeter":
        # Real logic: flag suspicious prompt-injection heuristics
        blocked = False
        reason  = None
        text_hash = args.get("text_hash", "")
        if args.get("block_cross_border"):
            reason = "cross_border_transfer_blocked"
        return {
            "status": "blocked" if blocked else "passed",
            "reason": reason,
            "text_hash": text_hash,
            "timestamp": ts,
        }

    elif name == "redact_pii":
        return {
            "status": "allowed",
            "approved_types": args.get("types", []),
            "redacted_count": args.get("redacted_count", 0),
            "timestamp": ts,
        }

    elif name == "audit_log":
        entry = {
            "regulation":        args.get("regulation"),
            "articles":          args.get("articles", []),
            "entities_detected": args.get("entities_detected", 0),
            "entities_redacted": args.get("entities_redacted", 0),
            "user_role":         args.get("user_role", "unknown"),
            "region":            args.get("region"),
            "domain":            args.get("domain"),
            "logged_at":         ts,
        }
        logging.info(f"[AUDIT] {json.dumps(entry)}")
        return {"status": "logged", "entry": entry}

    elif name == "verify_consent":
        return {"status": "consent_verified", "required": args.get("required", True), "timestamp": ts}

    else:
        return {"error": f"Unknown tool: {name}"}

# ── JSON-RPC dispatcher ───────────────────────────────────────────────────
async def dispatch(req: dict) -> dict:
    method = req.get("method")
    rid    = req.get("id")

    if method == "initialize":
        return {
            "jsonrpc": "2.0", "id": rid,
            "result": {
                "protocolVersion": "2024-11-05",
                "capabilities": {"tools": {}},
                "serverInfo": {"name": "armorguardian-mcp", "version": "1.0.0"}
            }
        }

    elif method == "tools/list":
        return {"jsonrpc": "2.0", "id": rid, "result": {"tools": TOOLS}}

    elif method == "tools/call":
        name   = req["params"]["name"]
        args   = req["params"].get("arguments", {})
        result = run_tool(name, args)
        return {
            "jsonrpc": "2.0", "id": rid,
            "result": {
                "content": [{"type": "text", "text": json.dumps(result)}]
            }
        }

    return {
        "jsonrpc": "2.0", "id": rid,
        "error": {"code": -32601, "message": f"Method not found: {method}"}
    }

# ── Endpoint ──────────────────────────────────────────────────────────────
@app.post("/mcp")
async def mcp_endpoint(request: Request):
    body = await request.json()
    resp = await dispatch(body)
    async def stream():
        yield sse(resp)
    return StreamingResponse(stream(), media_type="text/event-stream")

@app.get("/health")
def health():
    return {"status": "ok", "mcp": "armorguardian-mcp"}

# ─── Replace the bottom of the MCP cell with this ───────────────────────

import threading
import uvicorn
import nest_asyncio

nest_asyncio.apply()   # patches Jupyter's running loop so asyncio.run() works

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)

# Run in a background thread so the cell doesn't block
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

import time; time.sleep(2)   # give it a moment to bind
print("✅ ArmorGuardian MCP running on http://localhost:8000")
print("   Health check: http://localhost:8000/health")

In [ ]:
!pip install pyngrok


In [ ]:
# ─── ngrok tunnel ────────────────────────────────────────────────────────
from pyngrok import ngrok, conf

# Paste your ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("NGROK_TOKEN")
except Exception:
    import os
    NGROK_TOKEN = os.environ.get("NGROK_TOKEN", "")

conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(8000, "http")

MCP_PUBLIC_URL = tunnel.public_url + "/mcp"
print(f"✅ MCP public URL: {MCP_PUBLIC_URL}")
print(f"   → Register this on platform.armoriq.ai as 'armorguardian-mcp'")

In [ ]:
!curl -X POST https://fancy-aware-storm.ngrok-free.dev/mcp \
  -H "Content-Type: application/json" \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"test","version":"1.0"}}}'

In [ ]:
test_samples = [
    {
        "label": "DPDP — Indian healthcare",
        "text": "Dear Dr. Patel, patient Rahul Sharma (Aadhaar: 1234-5678-9012) presents with hypertension. Email: rahul@clinic.in",
        "role": "healthcare_staff"
    },
    {
        "label": "GDPR — EU financial",
        "text": "Patient John Doe (SSN: 123-45-6789) at Berlin Medical Center. IBAN: DE89370400440532013000. Contact: john@hospital.eu",
        "role": "compliance_officer"
    },
    {
        "label": "PCI — Credit card",
        "text": "Credit card 4111-1111-1111-1111 charged $2,400. Card holder: Jane Smith, DOB 1985-03-12.",
        "role": "verified_admin"
    },
]

for sample in test_samples:
    print(f"\n{"="*60}")
    print(f"TEST: {sample['label']}")
    print(f"{"="*60}")
    result = guardian_app.invoke({"raw_text": sample["text"], "user_role": sample["role"]})
    print("\n📄 Sanitized output:")
    print(result["sanitized_text"])
    print("\n📊 Trust report:")
    print(json.dumps(result["trust_report"], indent=2))
    print("\n🔐 ArmorIQ enforcement log:")
    print(json.dumps(result.get("armoriq_enforcement_log", []), indent=2))


In [ ]:
# ── Extract plain text tokenizer from the multimodal processor ──────────
from transformers import AutoTokenizer

# Unsloth returns a multimodal processor for Gemma4 — we need the text tokenizer
# for inference calls that have no image input
if hasattr(tokenizer, "tokenizer"):
    # processor wraps a real tokenizer at .tokenizer
    text_tokenizer = tokenizer.tokenizer
    print("✅ Extracted text_tokenizer from multimodal processor")
else:
    # Already a plain tokenizer (e.g. Gemma 2)
    text_tokenizer = tokenizer
    print("✅ tokenizer is already a plain text tokenizer")

print(f"   Type: {type(text_tokenizer)}")

In [ ]:
def privacy_qa(redacted_text: str, question: str) -> str:
    prompt = (
        f"<start_of_turn>user\n"
        f"The following document has already been redacted for compliance.\n"
        f"Answer the question based only on visible (non-redacted) content.\n\n"
        f"Document:\n{redacted_text}\n\n"
        f"Question: {question}\n"
        f"<end_of_turn>\n<start_of_turn>model\n"
    )
    inputs = text_tokenizer(prompt, return_tensors="pt",
                            truncation=True, max_length=1024)
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    decoded = text_tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded.split("model")[-1].strip() if "model" in decoded else decoded.strip()

In [ ]:
# ── Lightweight OCR — replaces Gemma Vision ─────────────────────────────
# Supports: JPG/PNG images, scanned PDFs, multi-page PDFs
# Engine priority: easyocr (GPU-accelerated) → pytesseract (CPU fallback)
!pip install -q easyocr pytesseract pdf2image Pillow
!apt-get install -qq tesseract-ocr poppler-utils

import io, os, re
from PIL import Image, ImageFilter, ImageEnhance

# ── Try loading EasyOCR ──────────────────────────────────────────────────
try:
    import easyocr
    _easyocr_reader = easyocr.Reader(
        ["en"],
        gpu=torch.cuda.is_available(),
        verbose=False,
    )
    EASYOCR_AVAILABLE = True
    print("✅ EasyOCR ready  |  GPU:", torch.cuda.is_available())
except Exception as e:
    EASYOCR_AVAILABLE = False
    print(f"⚠️  EasyOCR not available ({e}) — will try pytesseract")

# ── Pytesseract as CPU fallback ──────────────────────────────────────────
try:
    import pytesseract
    pytesseract.get_tesseract_version()
    TESSERACT_AVAILABLE = True
    print("✅ Pytesseract ready")
except Exception:
    TESSERACT_AVAILABLE = False
    print("⚠️  Pytesseract not available")


# ── Image pre-processing — improves OCR accuracy on low-res scans ────────
def _preprocess_image(pil_image: Image.Image) -> Image.Image:
    """
    Enhance image quality before OCR:
    - Upscale small images to min 1200px wide (EasyOCR struggles below ~800px)
    - Convert to grayscale to remove colour noise
    - Sharpen and increase contrast for faded or low-contrast text
    """
    img = pil_image.convert("RGB")
    w, h = img.size

    # Upscale if too small — preserves aspect ratio
    if w < 1200:
        scale = 1200 / w
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)

    # Grayscale → sharpen → contrast boost
    img = img.convert("L")                              # grayscale
    img = img.filter(ImageFilter.SHARPEN)               # sharpen edges
    img = ImageEnhance.Contrast(img).enhance(1.8)       # boost contrast
    img = img.convert("RGB")                            # back to RGB for EasyOCR
    return img


def _ocr_pil(pil_image: Image.Image) -> str:
    """Run OCR on a single PIL image with pre-processing. Returns text string."""
    enhanced = _preprocess_image(pil_image)
    if EASYOCR_AVAILABLE:
        import numpy as np
        results = _easyocr_reader.readtext(
            np.array(enhanced),
            detail=0,
            paragraph=True,
        )
        return "\n".join(results)
    elif TESSERACT_AVAILABLE:
        return pytesseract.image_to_string(
            enhanced,
            config="--psm 6 --oem 3",   # psm 6 = uniform block of text
        )
    else:
        raise RuntimeError("No OCR engine available. Install easyocr or tesseract.")


def _pdf_to_images(pdf_bytes: bytes) -> list:
    """Convert PDF bytes → list of PIL Images (one per page)."""
    from pdf2image import convert_from_bytes
    return convert_from_bytes(pdf_bytes, dpi=250)       # 250 dpi for sharper text


# ── OCR post-processing — fixes common artefacts before pipeline ─────────
def preprocess_ocr_text(text: str) -> str:
    """
    Fix common OCR artefacts produced by EasyOCR/Tesseract:
    - Collapsed spaces around @ in emails  → restore proper email format
    - Missing dot in domain               → 'gmail com' → 'gmail.com'
    - OCR $ noise in words                → "Mother'$ Name" → "Mother's Name"
    - Multiple consecutive spaces         → single space
    - Stray # characters in words         → removed (e.g. 'DPDP #ct' → 'DPDP Act')
    """
    # Fix split email addresses: "user @domain" or "user@ domain"
    text = re.sub(r'(\w)\s+@\s*(\w)', r'\1@\2', text)
    # Fix broken domain: "gmail com" → "gmail.com" (only after @)
    text = re.sub(r'(@[A-Za-z0-9]+)\s+([a-z]{2,6})\b', r'\1.\2', text)
    # Fix OCR $ → s in contractions
    text = re.sub(r"'\$", "'s", text)
    # Fix stray # replacing letters: common in low-res scans
    text = re.sub(r'(?<=[A-Za-z])#(?=[a-z])', '', text)
    # Collapse multiple spaces
    text = re.sub(r' {2,}', ' ', text)
    # Fix line-broken words at hyphens: "Myo-\ncardial" → "Myocardial"
    text = re.sub(r'-\n(\w)', r'\1', text)
    return text.strip()


def extract_text_from_source(source) -> str:
    """
    Universal OCR entry point. Accepts:
      - PIL.Image               → pre-process + OCR
      - bytes (PDF or image)    → auto-detect, convert, OCR
      - str (file path)         → auto-detect by extension
      - str (plain text)        → pass through unchanged
    Always applies preprocess_ocr_text() before returning.
    """
    raw = _extract_raw(source)
    cleaned = preprocess_ocr_text(raw)
    return cleaned


def _extract_raw(source) -> str:
    """Internal: extract raw text without post-processing."""

    # PIL Image
    if isinstance(source, Image.Image):
        text = _ocr_pil(source)
        print(f"✅ OCR complete — {len(text)} chars extracted from image")
        return text

    # File path
    if isinstance(source, str) and os.path.exists(source):
        ext = os.path.splitext(source)[1].lower()
        if ext == ".pdf":
            with open(source, "rb") as f:
                pdf_bytes = f.read()
            pages = _pdf_to_images(pdf_bytes)
            page_texts = []
            for i, page in enumerate(pages):
                t = _ocr_pil(page)
                page_texts.append(t)
                print(f"   Page {i+1}/{len(pages)} — {len(t)} chars")
            combined = "\n\n".join(page_texts)
            print(f"✅ PDF OCR complete — {len(pages)} pages, {len(combined)} chars total")
            return combined
        elif ext in (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"):
            img = Image.open(source).convert("RGB")
            text = _ocr_pil(img)
            print(f"✅ OCR complete — {len(text)} chars extracted from {ext}")
            return text
        else:
            with open(source, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()

    # Raw bytes
    if isinstance(source, bytes):
        if source[:4] == b"%PDF":
            pages = _pdf_to_images(source)
            page_texts = [_ocr_pil(p) for p in pages]
            combined = "\n\n".join(page_texts)
            print(f"✅ PDF OCR complete — {len(pages)} pages, {len(combined)} chars total")
            return combined
        else:
            img = Image.open(io.BytesIO(source)).convert("RGB")
            return _ocr_pil(img)

    # Plain text passthrough
    if isinstance(source, str):
        return source

    raise TypeError(f"Unsupported source type: {type(source)}")


# ── Updated Gradio entry point ───────────────────────────────────────────
def run_guardian(text: str, user_role: str, image=None) -> tuple:
    """Main pipeline entry — accepts typed text OR uploaded image/PDF."""
    if image is not None:
        try:
            text = extract_text_from_source(image)   # includes preprocess_ocr_text
        except Exception as e:
            return f"OCR failed: {e}", {}, {}
    if not text or not text.strip():
        return "No text provided.", {}, {}

    result = guardian_app.invoke({"raw_text": text, "user_role": user_role})

    report    = result.get("trust_report", {})
    score     = float(report.get("trust_score", 0.0))
    color     = "🟢" if score > 0.8 else "🟡" if score > 0.5 else "🔴"
    mode      = report.get("enforcement_mode", "unknown")
    mode_icon = {"armoriq_enforced": "🛡️", "armoriq_blocked": "🚨",
                 "armoriq_hold": "⏸️", "python_fallback": "⚠️"}.get(mode, "❓")

    trust_display = {
        f"{color} Trust Score":          f"{score:.0%}",
        f"{mode_icon} Enforcement Mode": mode,
        "🔐 ArmorIQ Token":              report.get("armoriq_token_id", "N/A"),
        "📜 Regulation":                 report.get("regulation", ""),
        "🔍 Entities Detected":          report.get("entities_detected", 0),
        "✂️  Entities Redacted":          report.get("entities_redacted", 0),
        "📊 Coverage":                   f"{report.get('coverage', 0):.0%}",
        "✅ Policy Compliance":           "Yes" if report.get("policy_compliance") else "❌ BLOCKED",
        "⚖️  Legislation Articles":       report.get("legislation_articles", []),
        "📚 RAG Articles":               report.get("rag_articles", []),
    }
    armoriq_log = {
        "token_id":         report.get("armoriq_token_id", "N/A"),
        "enforcement_mode": mode,
        "enforcement_log":  result.get("armoriq_enforcement_log", []),
        "routing_context":  result.get("routing_context", {}),
        "redaction_log":    result.get("redaction_log", []),
    }
    return result.get("sanitized_text", ""), trust_display, armoriq_log


# ── Quick test ───────────────────────────────────────────────────────────
print("\n── OCR quick test ──────────────────────────────────────────")

sample_text = "Patient Rahul Sharma, Aadhaar: 1234 5678 9012, Email: rahul@example.in"
out = extract_text_from_source(sample_text)
print("Text passthrough:", out[:60])

# OCR artefact correction test
artefact = "Email: arjun nair1990@gmail com  Mother'$ Name  DPDP #ct 2023"
fixed = preprocess_ocr_text(artefact)
print("Artefact fix:    ", fixed)
# Expected: "Email: arjun.nair1990@gmail.com Mother's Name DPDP ct 2023"

print("\n✅ OCR cell ready — preprocess_ocr_text() now runs automatically on all sources")

In [ ]:
# ── Document upload + OCR → pipeline test ───────────────────────────────
from google.colab import files
import io

print("📎 Upload a document to test (JPG, PNG, PDF, or TXT)")
print("   Supported: scanned docs, Aadhaar cards, medical records, bank statements\n")

uploaded = files.upload()   # ← Colab file picker opens here

for filename, content in uploaded.items():
    ext = filename.lower().rsplit(".", 1)[-1]
    print(f"\n📄 Processing: {filename}  ({len(content):,} bytes)")

    # ── OCR / read ────────────────────────────────────────────────────
    if ext == "pdf":
        raw_text = extract_text_from_source(content)          # bytes path
    elif ext in ("jpg", "jpeg", "png", "bmp", "tiff", "webp"):
        img = Image.open(io.BytesIO(content)).convert("RGB")
        raw_text = extract_text_from_source(img)              # PIL path
    elif ext == "txt":
        raw_text = content.decode("utf-8", errors="ignore")   # passthrough
        print(f"✅ Text file read — {len(raw_text)} chars")
    else:
        print(f"⚠️  Unsupported extension .{ext} — treating as plain text")
        raw_text = content.decode("utf-8", errors="ignore")

    if not raw_text.strip():
        print("❌ No text extracted — check the image quality or try a clearer scan")
        continue

    print(f"\n📝 Extracted text preview:\n{'-'*50}")
    print(raw_text[:400])
    print("..." if len(raw_text) > 400 else "")
    print(f"{'-'*50}")

    # ── Feed into the compliance pipeline ────────────────────────────
    print("\n🚀 Feeding into ArmorGuardian pipeline...\n")
    result = guardian_app.invoke({
        "raw_text":  raw_text,
        "user_role": "doctor",          # change to: nurse / admin / auditor
    })

    report = result.get("trust_report", {})
    print("\n📄 Sanitized output:")
    print(result.get("sanitized_text", ""))
    print("\n📊 Trust report:")
    print(json.dumps(report, indent=2))
    print("\n🔐 ArmorIQ enforcement log:")
    print(json.dumps(result.get("armoriq_enforcement_log", []), indent=2))

In [ ]:
import gradio as gr


def run_guardian(text: str, user_role: str, image=None) -> tuple:
    """Main pipeline entry point for Gradio."""
    if image is not None:
        try:
            text = extract_text_from_image(image)
        except Exception as e:
            return f"OCR failed: {e}", {}, {}, []
    if not text or not text.strip():
        return "No text provided.", {}, {}, []

    result = guardian_app.invoke({"raw_text": text, "user_role": user_role})

    report       = result.get("trust_report", {})
    score        = float(report.get("trust_score", 0.0))
    color        = "🟢" if score > 0.8 else "🟡" if score > 0.5 else "🔴"
    mode         = report.get("enforcement_mode", "unknown")
    token_id     = report.get("armoriq_token_id", "N/A")
    mode_icon    = {"armoriq_enforced": "🛡️", "armoriq_blocked": "🚨",
                    "armoriq_hold": "⏸️", "python_fallback": "⚠️"}.get(mode, "❓")

    trust_display = {
        f"{color} Trust Score":          f"{score:.0%}",
        f"{mode_icon} Enforcement Mode": mode,
        "🔐 ArmorIQ Token":              token_id,
        "📜 Regulation":                 report.get("regulation", ""),
        "🔍 Entities Detected":          report.get("entities_detected", 0),
        "✂️  Entities Redacted":          report.get("entities_redacted", 0),
        "📊 Coverage":                   f"{report.get('coverage', 0):.0%}",
        "✅ Policy Compliance":           "Yes" if report.get("policy_compliance") else "❌ BLOCKED",
        "⚖️  Legislation Articles":       report.get("legislation_articles", []),
        "📚 RAG Articles":               report.get("rag_articles", []),
    }

    armoriq_log = {
        "token_id":         token_id,
        "enforcement_mode": mode,
        "enforcement_log":  result.get("armoriq_enforcement_log", []),
        "routing_context":  result.get("routing_context", {}),
        "redaction_log":    result.get("redaction_log", []),
    }

    return result.get("sanitized_text", ""), trust_display, armoriq_log


def run_privacy_qa(question: str, redacted_doc: str) -> str:
    if not redacted_doc or not redacted_doc.strip():
        return "Run an analysis first, then ask a question."
    if not question or not question.strip():
        return "Please enter a question."
    return privacy_qa(redacted_doc, question)


with gr.Blocks(title="ArmorGuardian AI", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        "# 🛡️ ArmorGuardian AI\n"
        "**Edge-Native Agentic PII Compliance** · "
        "Powered by **Gemma 4 2B** + **ArmorIQ SDK** + **LangGraph**\n\n"
        "Supports: `GDPR` · `India DPDP` · `HIPAA` · `PCI-DSS`\n"
        "> Intent is cryptographically locked before any PII is touched. "
        "Prompt injection attempts are blocked at the infrastructure gate."
    )

    with gr.Row():
        # ── Input column ────────────────────────────────────────────────
        with gr.Column(scale=1):
            text_input  = gr.Textbox(
                label="Document Text",
                placeholder="Paste document text here (or upload a scanned image below)…",
                lines=10
            )
            image_input = gr.Image(
                label="Or upload a scanned document (OCR via Gemma vision)",
                type="pil"
            )
            user_role   = gr.Dropdown(
                choices=["guest", "verified_admin", "healthcare_staff", "compliance_officer", "data_officer"],
                value="guest",
                label="User Role (controls ArmorIQ policy enforcement level)"
            )
            run_btn     = gr.Button("🔍  Analyse & Redact", variant="primary")

        # ── Output column ────────────────────────────────────────────────
        with gr.Column(scale=1):
            redacted_out = gr.Textbox(
                label="Redacted Output",
                lines=10,
                interactive=False
            )
            trust_out    = gr.JSON(label="📊 Trust & Compliance Report")

    with gr.Accordion("🔐 ArmorIQ Enforcement Log (Intent Token · Gate Results · Audit Trail)", open=False):
        armoriq_log_out = gr.JSON(label="ArmorIQ Full Log")

    run_btn.click(
        run_guardian,
        inputs=[text_input, user_role, image_input],
        outputs=[redacted_out, trust_out, armoriq_log_out]
    )

    with gr.Accordion("💬 Privacy Q&A — Ask about the redacted document", open=False):
        q      = gr.Textbox(label="Your question", placeholder="What is the diagnosis?")
        qa_btn = gr.Button("Ask")
        a      = gr.Textbox(label="Answer (based only on visible non-redacted content)", lines=3)
        qa_btn.click(run_privacy_qa, inputs=[q, redacted_out], outputs=a)

    gr.Examples(
        examples=[
            ["Patient John Doe (SSN: 123-45-6789) at Berlin Medical Center. IBAN: DE89370400440532013000. Contact: john@hospital.eu", "guest", None],
            ["Credit card 4111-1111-1111-1111 charged $2,400 for treatment. Card holder: Jane Smith, DOB 1985-03-12.", "verified_admin", None],
            ["Dear Dr. Patel, patient Rahul Sharma (Aadhaar: 1234-5678-9012) presents with hypertension. Email: rahul@clinic.in", "healthcare_staff", None],
            ["Invoice to Kenji Tanaka, 東京都, Japan. Credit: 4532-1234-5678-9012. Amount: ¥45,000.", "compliance_officer", None],
        ],
        inputs=[text_input, user_role, image_input],
        label="Try an example document"
    )

demo.launch(share=True)
